# Food Waste Analysis â€” Lookup Preparation and Spark Results

Run this notebook first to create the two lookup CSV files required by `spark/jobs/food_waste_processing_main.py`. Then run the Spark processing job. Finally, rerun the **Gold analysis tables** and **Dashboard CSV** cells below to display the new outputs.

The setup uses the existing PostgreSQL source tables and the retained setup/views block in `db/queries/food_waste_analysis.sql`.

In [2]:

from pathlib import Path
import os
import re
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sqlalchemy.engine import make_url
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE

SQL_FILE = PROJECT_ROOT / "db" / "queries" / "food_waste_analysis.sql"
LOOKUP_DIR = PROJECT_ROOT / "data" / "lookup"
GOLD_DIR = PROJECT_ROOT / "data" / "processed_by_spark" / "gold"
DASHBOARD_CSV = PROJECT_ROOT / "data" / "processed_by_spark" / "dashboard" / "food_waste_dashboard.csv"

# Load database settings from the project .env file
load_dotenv(PROJECT_ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL not found in .env")

db_url = make_url(DATABASE_URL)

DB_CONFIG = {
    "host": db_url.host,
    "port": db_url.port,
    "dbname": db_url.database,
    "user": db_url.username,
    "password": db_url.password,
}

if not SQL_FILE.exists():
    raise FileNotFoundError(f"Setup SQL not found: {SQL_FILE}")

LOOKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT_ROOT}")
print(f"Lookup output: {LOOKUP_DIR}")
print(f"Database: {DB_CONFIG['dbname']}")
print(f"Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")


Project: C:\Users\yarag\Documents\GitHub\Food-Waste
Lookup output: C:\Users\yarag\Documents\GitHub\Food-Waste\data\lookup
Database: food_waste_db
Host: postgres:5432


## 1. Create lookup tables and CSV files

This cell recreates the shared SQL lookup/views, then exports the two lookup tables to `data/lookup/`. Spark reads these same files.

In [ ]:
sql_text = SQL_FILE.read_text(encoding="utf-8-sig")
setup_sql = re.sub(r"(?m)^--\s*(?:name|page|title|question|visual):.*(?:\n|$)", "", sql_text).strip()
connection = psycopg2.connect(**DB_CONFIG)
try:
    with connection.cursor() as cursor:
        cursor.execute(setup_sql)
    connection.commit()

    country_lookup = pd.read_sql_query(
        "SELECT LTRIM(m49_code, '0') AS m49_code, country_name, continent "
        "FROM country_continent ORDER BY continent, country_name",
        connection,
        dtype={"m49_code": str},
    )
    activity_lookup = pd.read_sql_query(
        "SELECT activity_name, stage_group FROM activity_stage_map "
        "ORDER BY activity_name",
        connection,
    )

    country_path = LOOKUP_DIR / "country_continent.csv"
    activity_path = LOOKUP_DIR / "activity_stage_map.csv"
    country_lookup.to_csv(country_path, index=False)
    activity_lookup.to_csv(activity_path, index=False)
    print(f"Created {country_path} ({len(country_lookup):,} rows)")
    print(f"Created {activity_path} ({len(activity_lookup):,} rows)")
finally:
    connection.close()

## 2. Review the generated lookup CSVs

In [ ]:
lookups = {
    "country_continent": pd.read_csv(LOOKUP_DIR / "country_continent.csv", dtype={"m49_code": str}),
    "activity_stage_map": pd.read_csv(LOOKUP_DIR / "activity_stage_map.csv"),
}
for name, frame in lookups.items():
    print(f"{name}: {len(frame):,} rows")
    display(frame.head(10))

## 3. Gold analysis tables

Run `food_waste_processing_main.py` after creating the lookups, then run this cell to load and display the new Gold CSV tables.

In [ ]:
tables = {}
if not GOLD_DIR.exists():
    print(f"Gold outputs are not present yet: {GOLD_DIR}. Run the Spark processing job, then rerun this cell.")
else:
    for table_dir in sorted(path for path in GOLD_DIR.iterdir() if path.is_dir()):
        part_files = sorted(table_dir.glob("part-*.csv"))
        if not part_files:
            continue
        frames = [pd.read_csv(file) for file in part_files]
        tables[table_dir.name] = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
        print(f"{table_dir.name}: {len(tables[table_dir.name]):,} rows")
        display(tables[table_dir.name].head(20))

## 4. Dashboard CSV

In [ ]:
if DASHBOARD_CSV.exists():
    dashboard = pd.read_csv(DASHBOARD_CSV)
    print(f"Dashboard: {len(dashboard):,} rows, {len(dashboard.columns)} columns")
    display(dashboard.head(20))
else:
    dashboard = None
    print(f"Dashboard CSV not found yet: {DASHBOARD_CSV}. Run the Spark job, then rerun this cell.")